<a href="https://colab.research.google.com/github/ALiao18/SUDC/blob/main/autoML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AutoML: Prone Positioning Prediction (SUDC)

**Target:** `child_face_position_fi___1`
**Metric:** ROC-AUC, 10-fold bagged CV
**Library:** AutoGluon TabularPredictor

## Feature set rationale

Feature sets below are now loaded dynamically
from `interpretable_modeling_phase2_stable_features.csv` (Phase 2's actual,
current nested-CV output), so this notebook can't silently drift out of sync
with the ML pipeline again.

| Run | Features | Rationale |
|-----|----------|-----------|
| 1 | 15 (p<0.1 stable core) | Selected in every one of 10 folds at the more permissive threshold |
| 2 | 8 (p<0.05 stable core) | Selected in every one of 10 folds at the stricter threshold |
| 3 | Run 2 + 2 engineered  | Tests whether interactions add signal in tree ensembles |

**Anchor note:** `febrile_sz___any_type_Final` is present in Run 1 but **not**
Run 2/3 -- it's stable at p<0.1 but not p<0.05 (already reported to Laura/Dr.
Chen). This is expected, not a bug; no anchor-forcing logic is applied here
(consistent with its removal elsewhere in the ML pipeline).

**Low-prevalence caveat:** both stable sets include `child_Atrial_Septal_Defect_MH`
and `in_hospital_newborn_cyanosis` (n=4 positive cases each). These can produce
artificially strong importance/LRT signal via near-complete separation at this
sample size rather than real effect (see ML.ipynb Phase 5 caveat). Retained
here for consistency with the ML pipeline's own decision to flag rather than
drop them -- interpret their AutoML importance ranking with that in mind.


## setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install autogluon --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.5/259.5 kB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.6/227.6 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.9/98.9 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 452.1/452.1 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.8/244.8 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 kB 6.3 MB/s eta 0:00

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import os
from sklearn.metrics import roc_auc_score
from autogluon.tabular import TabularPredictor

In [ ]:
# ── constants ─────────────────────────────────────────────────
TARGET      = 'child_face_position_fi___1'
ANCHOR      = 'febrile_sz___any_type_Final'
FILE_PATH   = # insert file save path
BASE_OUTPUT = # insert automl results save path
save_path = # insert output save path

# ── run parameters ────────────────────────────────
TIME_LIMIT = 1800
PRESET     = 'best_quality'
N_BAG_FOLDS    = 10
N_STACK_LEVELS = 2

# ── autogluon kwargs ────────
AG_KWARGS = dict(
    label       = TARGET,
    eval_metric = 'roc_auc',
)
FIT_KWARGS = dict(
    presets              = PRESET,
    num_bag_folds        = N_BAG_FOLDS,
    num_stack_levels     = N_STACK_LEVELS,
    time_limit           = TIME_LIMIT,
    feature_prune_kwargs = {'force_prune': False},
)

# ── helpers ───────────────────────────────────────────────────
def get_leaderboard(predictor):
    """
    OOF-based leaderboard only — never pass training data.
    score_val = honest OOF AUC from 10-fold bagging.
    score_test is meaningless here (no holdout set; has_val=False).
    """
    return (predictor.leaderboard(silent=True)
                     .sort_values('score_val', ascending=False)
                     .reset_index(drop=True))

def get_best_base(lb):
    """
    Best non-ensemble, non-neural-net, non-LightGBMXT model by OOF AUC.
    NeuralNet and LightGBMXT excluded due to inflated OOF estimates at small n.
    """
    exclude = lb['model'].str.contains('Ensemble|Weighted|NeuralNet|LightGBMXT')
    return lb[~exclude].iloc[0]

def get_importance(predictor, df):
    """Return permutation importance dataframe."""
    return predictor.feature_importance(df)

def anchor_rank(imp, feature=ANCHOR):
    """Return (rank, importance) for a feature in importance df."""
    if feature not in imp.index:
        return None, None
    return imp.index.get_loc(feature) + 1, imp.loc[feature, 'importance']


In [ ]:
# ── load and engineer data ────────────────────────────────────
df_raw = pd.read_csv(FILE_PATH)
df_raw = df_raw.drop(columns=['sudcrrc_id'], errors='ignore')

df_eng = df_raw.copy()

# Engineered interactions.
df_eng['fs_x_crank']   = df_eng[ANCHOR] * df_eng['last_48hr_crankiness']
df_eng['age_x_female'] = df_eng['age_sudc'] * df_eng['female']

ENG_COLS = ['fs_x_crank', 'age_x_female']
num_eng_cols = len(ENG_COLS)

# ── feature sets ──────────────────────────────────────────────
STABLE_FEATURES_PATH = os.path.join(save_path, 'interpretable_modeling_phase2_stable_features.csv')
stable_df = pd.read_csv(STABLE_FEATURES_PATH)

def load_stable_set(threshold):
    feats = stable_df.loc[stable_df['p_threshold'] == threshold, 'feature'].tolist()
    missing = [f for f in feats if f not in df_eng.columns]
    if missing:
        print(f"  [WARNING] p<{threshold} stable set: {len(missing)} feature(s) "
              f"in the CSV are not in df_eng and will be dropped: {missing}")
    return [f for f in feats if f in df_eng.columns]

# Run 1: p<0.1 stable core (15 features, selected in every fold in Phase 2)
FEATURES_RUN1 = load_stable_set(0.1)

# Run 2: p<0.05 stable core (8 features, selected in every fold in Phase 2)
# NOTE: febrile_sz___any_type_Final (ANCHOR) is NOT in this set -- it's
# stable at p<0.1 but not p<0.05
FEATURES_RUN2 = load_stable_set(0.05)

# Run 3: Run 2 + the 2 reconstructible engineered interactions
FEATURES_RUN3 = list(dict.fromkeys(
    FEATURES_RUN2 + [f for f in ENG_COLS if f in df_eng.columns]
))

num_feat_r1 = len(FEATURES_RUN1)
num_feat_r2 = len(FEATURES_RUN2)
num_feat_r3 = len(FEATURES_RUN3)

LOW_PREVALENCE_CAVEAT = ['child_Atrial_Septal_Defect_MH', 'in_hospital_newborn_cyanosis']

# ── confirm setup ─────────────────────────────────────────────
y = df_eng[TARGET]
print(f"Dataset: {df_eng.shape}, prone_rate={y.mean():.3f}")
print(f"\nFeature counts (excl. target):")
print(f"  Run 1 (p<0.1 stable):              {num_feat_r1} features")
print(f"  Run 2 (p<0.05 stable):              {num_feat_r2} features")
print(f"  Run 3 (Run 2 + {num_eng_cols} engineered):     {num_feat_r3} features")
print(f"\nRun parameters:")
print(f"  preset:           {PRESET}")
print(f"  time_limit:       {TIME_LIMIT}s per run  "
      f"({TIME_LIMIT * 3 / 60:.0f} min total)")
print(f"  num_bag_folds:    {N_BAG_FOLDS}")
print(f"  num_stack_levels: {N_STACK_LEVELS}")
print(f"\n{ANCHOR} present in:")
for run_name, feats in [('Run 1', FEATURES_RUN1), ('Run 2', FEATURES_RUN2), ('Run 3', FEATURES_RUN3)]:
    print(f"  {run_name}: {'yes' if ANCHOR in feats else 'no (expected -- not stable at p<0.05)'}")
print(f"\nLow-prevalence caveat features present in:")
for run_name, feats in [('Run 1', FEATURES_RUN1), ('Run 2', FEATURES_RUN2), ('Run 3', FEATURES_RUN3)]:
    present = [f for f in LOW_PREVALENCE_CAVEAT if f in feats]
    print(f"  {run_name}: {present if present else 'none'}")


Dataset: (317, 187), prone_rate=0.552

Feature counts (excl. target):
  Run 1 (p<0.1 stable):              15 features
  Run 2 (p<0.05 stable):              8 features
  Run 3 (Run 2 + 2 engineered):     10 features

Run parameters:
  preset:           best_quality
  time_limit:       1800s per run  (90 min total)
  num_bag_folds:    10
  num_stack_levels: 2

febrile_sz___any_type_Final present in:
  Run 1: yes
  Run 2: no (expected -- not stable at p<0.05)
  Run 3: no (expected -- not stable at p<0.05)

Low-prevalence caveat features present in:
  Run 1: ['child_Atrial_Septal_Defect_MH', 'in_hospital_newborn_cyanosis']
  Run 2: ['child_Atrial_Septal_Defect_MH', 'in_hospital_newborn_cyanosis']
  Run 3: ['child_Atrial_Septal_Defect_MH', 'in_hospital_newborn_cyanosis']


# run

In [ ]:
results = {}

for run_name, features, run_id in [
    (f'Run 1 — {num_feat_r1} features (p<0.1 stable core)',                    FEATURES_RUN1, 'run1'),
    (f'Run 2 — {num_feat_r2} features (p<0.05 stable core)',                   FEATURES_RUN2, 'run2'),
    (f'Run 3 — {num_feat_r2} (p<0.05 core) + {num_eng_cols} engineered',       FEATURES_RUN3, 'run3'),
]:
    print("\n" + "="*70)
    print(run_name.upper())
    print("="*70)

    df_run = df_eng[features + [TARGET]].copy()
    print(f"Input: {df_run.shape[0]} rows, {df_run.shape[1]-1} features")

    # --- Fit ---
    predictor = TabularPredictor(
        path=f"{BASE_OUTPUT}/{run_id}",
        **AG_KWARGS
    ).fit(train_data=df_run, **FIT_KWARGS)

    # --- Leaderboard ---
    lb   = get_leaderboard(predictor)
    best = get_best_base(lb)
    print(f"\nLeaderboard (top 10):")
    print(f"{'Model':<45} {'val_AUC':>9}")
    print("-" * 56)
    for _, row in lb.head(10).iterrows():
        marker = ' ←' if row['model'] == best['model'] else ''
        print(f"  {row['model']:<43} {row['score_val']:>9.4f}{marker}")
    print(f"\n  Best base model:   {best['model']}")
    print(f"  Best base val AUC: {best['score_val']:.4f}")

    # --- Importance ---
    imp = get_importance(predictor, df_run)

    rank, val = anchor_rank(imp)
    if rank:
        print(f"\n  {ANCHOR}: rank {rank} of {len(imp)}, importance={val:.5f}")
    else:
        print(f"\n  WARNING: {ANCHOR} not found in importance rankings")

    # --- Save leaderboard ---
    lb_save = lb.copy()
    lb_save.insert(0, 'run_id',   run_id)
    lb_save.insert(1, 'run_name', run_name)
    lb_save.to_csv(
        os.path.join(save_path, f'automl_{run_id}_leaderboard.csv'),
        index=False
    )

    # --- Save importance (all runs) ---
    imp_save = imp.copy().reset_index()
    imp_save.columns = ['feature'] + list(imp_save.columns[1:])
    imp_save.insert(0, 'run_id',   run_id)
    imp_save.insert(1, 'run_name', run_name)
    imp_save['anchor_rank'] = rank
    imp_save['n_features']  = len(features)
    imp_save.to_csv(
        os.path.join(save_path, f'automl_{run_id}_importance.csv'),
        index=False
    )

    # --- Save importance for Run 2 ---

    # --- Save per-run summary row ---
    summary_row = {
        'run_id':            run_id,
        'run_name':          run_name,
        'n_features':        len(features),
        'n_rows':            df_run.shape[0],
        'best_model':        best['model'],
        'best_oof_auc':      best['score_val'],
        'ensemble_val_auc':  lb.iloc[0]['score_val'],
        'anchor_rank':       rank,
        'anchor_importance': val,
        'preset':            PRESET,
        'time_limit':        TIME_LIMIT,
        'n_bag_folds':       N_BAG_FOLDS,
        'n_stack_levels':    N_STACK_LEVELS,
    }

    # --- Store in results ---
    results[run_id] = {
        'name':        run_name,
        'features':    features,
        'predictor':   predictor,
        'lb':          lb,
        'best_base':   best,
        'imp':         imp,
        #'shap':        shap_vals,
        'summary_row': summary_row,
    }

    print(f"\n  ✓ {run_name} complete")

# --- Save combined summary across all runs ---
summary_df = pd.DataFrame([results[r]['summary_row'] for r in ['run1','run2','run3']])
summary_df.to_csv(
    os.path.join(save_path, 'automl_all_runs_summary.csv'),
    index=False
)

print("\n" + "="*70)
print("ALL RUNS COMPLETE")
print("="*70)
print(f"\nFiles saved to {save_path}:")
print(f"  automl_run1_leaderboard.csv")
print(f"  automl_run2_leaderboard.csv")
print(f"  automl_run3_leaderboard.csv")
print(f"  automl_run1_importance.csv")
print(f"  automl_run2_importance.csv")
print(f"  automl_run3_importance.csv")
print(f"  automl_all_runs_summary.csv  (bootstrap columns added by next cell)")
print(f"\nRun the next cell (Bootstrap confidence intervals) to add uncertainty estimates.")



RUN 1 — 15 FEATURES (P<0.1 STABLE CORE)
Input: 317 rows, 15 features


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Apr 30 18:17:14 UTC 2026
CPU Count:          2
Pytorch Version:    2.9.1+cu128
CUDA Version:       CUDA is not available
Memory Avail:       11.03 GB / 12.67 GB (87.0%)
Disk Space Avail:   72.30 GB / 107.72 GB (67.1%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=2, num_bag_folds=10, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or disable stacking as a consequence.
	This is used to identify the optimal `num_stack_levels`

(_ray_fit pid=8938) [1000]	valid_set's binary_logloss: 0.463904
(_ray_fit pid=9158) [1000]	valid_set's binary_logloss: 0.472823 [repeated 2x across cluster] (Ray deduplicates logs by default. Set RAY_DEDUP_LOGS=0 to disable log deduplication, or see https://docs.ray.io/en/master/ray-observability/user-guides/configure-logging.html#log-deduplication for more options.)


(_dystack pid=8619) 	0.724	 = Validation score   (roc_auc)
(_dystack pid=8619) 	41.18s	 = Training   runtime
(_dystack pid=8619) 	0.05s	 = Validation runtime
(_dystack pid=8619) Fitting model: LightGBM_BAG_L1 ... Training model for up to 138.41s of the 375.23s of remaining time.
(_dystack pid=8619) 	Fitting 10 child models (S1F1 - S1F10) | Fitting with ParallelLocalFoldFittingStrategy (2 workers, per: cpus=1, gpus=0, memory=0.19%)


(_ray_fit pid=9528) [1000]	valid_set's binary_logloss: 0.372977


...


  ✓ Run 2 — 8 features (p<0.05 stable core) complete

RUN 3 — 8 (P<0.05 CORE) + 2 ENGINEERED
Input: 317 rows, 10 features


Leaderboard on holdout data (DyStack):
                      model  score_holdout  score_val eval_metric  pred_time_test  pred_time_val    fit_time  pred_time_test_marginal  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0       WeightedEnsemble_L3       0.796875   0.839068     roc_auc        2.284909       0.625227  263.420987                 0.005888                0.000634           0.025462            3       True         15
1           CatBoost_BAG_L2       0.787500   0.836098     roc_auc        2.207208       0.594910  222.156372                 0.097660                0.053324          48.337669            2       True         14
2   RandomForestEntr_BAG_L2       0.785937   0.786073     roc_auc        2.211685       0.767873  174.642892                 0.102136                0.226287           0.824190            2       True         13
3           LightGBM_BAG_L2       0.784375   0.827496     roc_auc        2.181362       0.571269  215.057855     


Leaderboard (top 10):
Model                                           val_AUC
--------------------------------------------------------
  WeightedEnsemble_L4                            0.8154
  WeightedEnsemble_L3                            0.8112
  NeuralNetFastAI_BAG_L3                         0.8095
  WeightedEnsemble_L2                            0.8033
  CatBoost_BAG_L3                                0.8025 ←
  LightGBMXT_BAG_L3                              0.8006
  NeuralNetTorch_BAG_L2                          0.7990
  LightGBM_BAG_L3                                0.7971
  NeuralNetFastAI_BAG_L2                         0.7967
  RandomForestEntr_BAG_L2                        0.7943

  Best base model:   CatBoost_BAG_L3
  Best base val AUC: 0.8025


	192.84s	= Expected runtime (38.57s per shuffle set)
	37.65s	= Actual runtime (Completed 5 of 5 shuffle sets)




  ✓ Run 3 — 8 (p<0.05 core) + 2 engineered complete

ALL RUNS COMPLETE

Files saved to /content/drive/MyDrive/S25/Langone/Febrile_Seizure_Prediction/Code/final/fig&supplement_july:
  automl_run1_leaderboard.csv
  automl_run2_leaderboard.csv
  automl_run3_leaderboard.csv
  automl_run1_importance.csv
  automl_run2_importance.csv
  automl_run3_importance.csv
  automl_all_runs_summary.csv  (bootstrap columns added by next cell)

Run the next cell (Bootstrap confidence intervals) to add uncertainty estimates.


## Bootstrap confidence intervals
Run this after Cell 8 (main fit loop) completes in the same session, or after
re-mounting Drive + reinstalling autogluon + reloading saved predictors from
disk in a fresh session.

Replaces the deprecated contiguous-chunk "fold AUC recovery" method. This cell instead computes a percentile bootstrap
over the pooled out-of-fold predictions: resample rows with replacement,
recompute AUC, repeat 1000x, and take percentiles of the resulting
distribution as the confidence interval.


In [ ]:
# ── Bootstrap confidence intervals (added 2026-08, replaces contiguous-chunk method) ──
import numpy as np

N_BOOT   = 1000
BOOT_SEED = 42
rng = np.random.RandomState(BOOT_SEED)

summary_df = pd.read_csv(os.path.join(save_path, 'automl_all_runs_summary.csv'))

# Only the target column is needed here (to align with the OOF index) -- no
# need to reconstruct df_eng's engineered feature columns.
df_raw_recovery = pd.read_csv(FILE_PATH)
df_raw_recovery = df_raw_recovery.drop(columns=['sudcrrc_id'], errors='ignore')
y_full = df_raw_recovery[TARGET]

bootstrap_summary_rows = []

for _, row in summary_df.iterrows():
    run_id     = row['run_id']
    best_model = row['best_model']
    print(f"\n{'='*70}\n{run_id}: bootstrapping OOF AUC for {best_model}\n{'='*70}")

    # Prefer the in-memory predictor from Cell 8's `results` dict if this is
    # the same session; fall back to loading from disk otherwise (e.g. after
    # a Colab disconnect/remount).
    if 'results' in dir() and run_id in results:
        predictor = results[run_id]['predictor']
    else:
        predictor = TabularPredictor.load(f"{BASE_OUTPUT}/{run_id}")
        # If this raises a version-mismatch error (e.g. reloading in a fresh
        # session with a different autogluon version), retry with:
        #   TabularPredictor.load(f"{BASE_OUTPUT}/{run_id}", require_version_match=False)

    try:
        # as_multiclass=False -> Series of positive-class probability,
        # matching what roc_auc_score expects for binary y.
        oof_proba = predictor.predict_proba_oof(model=best_model, as_multiclass=False)
        y_oof     = y_full.loc[oof_proba.index]

        n = len(oof_proba)
        y_arr = y_oof.values
        p_arr = oof_proba.values

        oof_point_estimate = roc_auc_score(y_arr, p_arr)

        boot_aucs = []
        for _b in range(N_BOOT):
            idx = rng.randint(0, n, size=n)
            y_b, p_b = y_arr[idx], p_arr[idx]
            if len(np.unique(y_b)) < 2:
                continue  # guard against a degenerate resample (one class only)
            boot_aucs.append(roc_auc_score(y_b, p_b))

        boot_aucs = np.array(boot_aucs)
        boot_median  = float(np.median(boot_aucs))
        boot_q1      = float(np.percentile(boot_aucs, 25))
        boot_q3      = float(np.percentile(boot_aucs, 75))
        boot_ci_lo   = float(np.percentile(boot_aucs, 2.5))
        boot_ci_hi   = float(np.percentile(boot_aucs, 97.5))

        boot_df = pd.DataFrame({
            'run_id':         run_id,
            'bootstrap_iter': range(len(boot_aucs)),
            'bootstrap_auc':  boot_aucs,
        })
        boot_df.to_csv(
            os.path.join(save_path, f'automl_{run_id}_bootstrap_auc.csv'),
            index=False
        )

        print(f"  OOF point estimate: {oof_point_estimate:.4f}")
        print(f"  Bootstrap median:   {boot_median:.4f}  (Q1={boot_q1:.4f}, Q3={boot_q3:.4f})")
        print(f"  95% CI:             [{boot_ci_lo:.4f}, {boot_ci_hi:.4f}]  ({len(boot_aucs)}/{N_BOOT} valid resamples)")

    except Exception as e:
        print(f"  FAILED: {type(e).__name__}: {e}")
        oof_point_estimate = None
        boot_median = boot_q1 = boot_q3 = boot_ci_lo = boot_ci_hi = None

    bootstrap_summary_rows.append({
        'run_id':               run_id,
        'oof_auc_point_estimate': oof_point_estimate,
        'bootstrap_median_auc': boot_median,
        'bootstrap_q1_auc':     boot_q1,
        'bootstrap_q3_auc':     boot_q3,
        'bootstrap_ci95_lo':    boot_ci_lo,
        'bootstrap_ci95_hi':    boot_ci_hi,
    })

# ── Patch the existing summary in place rather than creating a second file ──
boot_summary_df = pd.DataFrame(bootstrap_summary_rows).set_index('run_id')
summary_df = summary_df.set_index('run_id')
for col in ['oof_auc_point_estimate', 'bootstrap_median_auc', 'bootstrap_q1_auc',
            'bootstrap_q3_auc', 'bootstrap_ci95_lo', 'bootstrap_ci95_hi']:
    summary_df[col] = boot_summary_df[col]
summary_df = summary_df.reset_index()
summary_df.to_csv(os.path.join(save_path, 'automl_all_runs_summary.csv'), index=False)

print(f"\n{'='*70}\nPatched automl_all_runs_summary.csv with bootstrap CIs.\n{'='*70}")
print(summary_df[['run_id', 'best_oof_auc', 'oof_auc_point_estimate',
                   'bootstrap_median_auc', 'bootstrap_ci95_lo', 'bootstrap_ci95_hi']].to_string(index=False))



run1: bootstrapping OOF AUC for CatBoost_BAG_L3
  OOF point estimate: 0.8238
  Bootstrap median:   0.8228  (Q1=0.8080, Q3=0.8393)
  95% CI:             [0.7790, 0.8666]  (1000/1000 valid resamples)

run2: bootstrapping OOF AUC for CatBoost_BAG_L1
  OOF point estimate: 0.7417
  Bootstrap median:   0.7430  (Q1=0.7243, Q3=0.7614)
  95% CI:             [0.6847, 0.7969]  (1000/1000 valid resamples)

run3: bootstrapping OOF AUC for CatBoost_BAG_L3
  OOF point estimate: 0.8025
  Bootstrap median:   0.8033  (Q1=0.7876, Q3=0.8193)
  95% CI:             [0.7538, 0.8496]  (1000/1000 valid resamples)

Patched automl_all_runs_summary.csv with bootstrap CIs.
run_id  best_oof_auc  oof_auc_point_estimate  bootstrap_median_auc  bootstrap_ci95_lo  bootstrap_ci95_hi
  run1      0.823783                0.823783              0.822804           0.779029           0.866589
  run2      0.741690                0.741690              0.743037           0.684651           0.796920
  run3      0.802495           

In [ ]:
# ── Sanity check: reload and display the final patched summary ──
final_check = pd.read_csv(os.path.join(save_path, 'automl_all_runs_summary.csv'))
print("Final automl_all_runs_summary.csv columns:")
print(final_check.columns.tolist())
final_check


Final automl_all_runs_summary.csv columns:
['run_id', 'run_name', 'n_features', 'n_rows', 'best_model', 'best_oof_auc', 'ensemble_val_auc', 'anchor_rank', 'anchor_importance', 'preset', 'time_limit', 'n_bag_folds', 'n_stack_levels', 'oof_auc_point_estimate', 'bootstrap_median_auc', 'bootstrap_q1_auc', 'bootstrap_q3_auc', 'bootstrap_ci95_lo', 'bootstrap_ci95_hi']


,run_id,run_name,n_features,n_rows,best_model,best_oof_auc,ensemble_val_auc,anchor_rank,anchor_importance,preset,time_limit,n_bag_folds,n_stack_levels,oof_auc_point_estimate,bootstrap_median_auc,bootstrap_q1_auc,bootstrap_q3_auc,bootstrap_ci95_lo,bootstrap_ci95_hi
0,run1,Run 1 — 15 features (p<0.1 stable core),15,317,CatBoost_BAG_L3,0.823783,0.831388,1.0,0.069034,best_quality,1800,10,2,0.823783,0.822804,0.807979,0.839336,0.779029,0.866589
1,run2,Run 2 — 8 features (p<0.05 stable core),8,317,CatBoost_BAG_L1,0.741690,0.766519,NaN,NaN,best_quality,1800,10,2,0.741690,0.743037,0.724258,0.761379,0.684651,0.796920
2,run3,Run 3 — 8 (p<0.05 core) + 2 engineered,10,317,CatBoost_BAG_L3,0.802495,0.815412,NaN,NaN,best_quality,1800,10,2,0.802495,0.803314,0.787641,0.819297,0.753773,0.849623
